# 02 — Manual Double Machine Learning walkthrough

This notebook reconstructs the primary DML-PLR estimator from held-out nuisance predictions instead of calling a DML package. For $\ell_0(X)=E[Y|X]$ and $m_0(X)=E[D|X]$, define residuals $\tilde Y=Y-\hat\ell(X)$ and $\tilde D=D-\hat m(X)$. The estimator is $\hat\theta=\sum_i\tilde D_i\tilde Y_i/\sum_i\tilde D_i^2$. Each nuisance prediction is generated out of fold.

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from dml_health_benchmark.dgp import SCENARIOS, generate_data
from dml_health_benchmark.learners import make_lasso_regressor, make_lasso_propensity


In [ ]:
data=generate_data(SCENARIOS['A'], seed=123)
X,d,y=data.X,data.d,data.y
n=len(y); l_hat=np.full(n,np.nan); m_hat=np.full(n,np.nan)
split=StratifiedKFold(5,shuffle=True,random_state=123)
for fold,(tr,te) in enumerate(split.split(X,d)):
    ly=make_lasso_regressor(1000+fold); md=make_lasso_propensity(2000+fold)
    ly.fit(X[tr],y[tr]); md.fit(X[tr],d[tr])
    l_hat[te]=ly.predict(X[te]); m_hat[te]=md.predict_proba(X[te])[:,1]
y_res=y-l_hat; d_res=d-m_hat
theta_hat=np.sum(d_res*y_res)/np.sum(d_res**2)
score=d_res*(y_res-theta_hat*d_res); jac=np.mean(d_res**2)
influence=score/jac; se=np.std(influence,ddof=1)/np.sqrt(n)
{'true_theta':1.0,'theta_hat':theta_hat,'std_error':se}


## Why orthogonality matters

The score reduces first-order sensitivity to nuisance-estimation error, but nuisance quality still matters. Scenario C is designed to demonstrate that raw-linear nuisance DML can remain biased when the learner cannot represent nonlinear confounding, while the same orthogonal-score procedure performs well with an adequate rich nuisance representation.